# predx: end-to-end test of the 510(k) extraction pipeline on free Colab

Tests the **existing** pipeline (fetch → text/OCR → section + predicate parsing → dataset build → audit) in six layers, cheapest first.
No Google Cloud project, billing or API credits are needed. Colab free tier = ~2 vCPU, ~12 GB RAM, ~100 GB temporary disk.

| # | test | needs internet | time (free Colab) | what a PASS means |
|---|------|----------------|-------------------|-------------------|
| 1 | unit tests (74) | no | ~3 min | code works in this environment |
| 2 | synthetic end-to-end demo | no | ~2 min | all stages + output files + audit wire together |
| 3 | **10 real FDA PDFs** shipped in the repo, with hand-verified predicates | no | ~1 min | parsing real documents still gives the right predicates |
| 4 | OCR stress curve (synthetic scans) | no | ~4 min | OCR settings hold up on degraded scans |
| 5 | **live FDA probe**: downloads PDFs you list, parses them | **yes** | ~5 min | fetcher works against the real server; parser generalises |
| 6 | **mini pilot**: openFDA metadata → one specialty → fetch → parse → build → audit, plus a hand check | **yes** | ~40-50 min (400 subjects) | the whole ETL works on real data, with the audit's go / no-go |

**How to use:** run the cells top to bottom (Runtime → Run all works, or go one cell at a time). Cells 5-6 skip themselves if the FDA is unreachable.
You will be asked **once** to upload the repo zip, or set `REPO_GIT_URL` in the settings cell to clone it from GitHub instead.
Free Colab disconnects after ~90 min idle and ~12 h total; every pipeline stage is resumable, so after a disconnect re-run from the top and it picks up from the cache **only if the files survived** (they live on the temporary disk; use the Drive cell at the end to keep results).

## 0. Settings (edit, then run)

In [ ]:
CONTACT_EMAIL = "you@columbia.edu"      # goes into the User-Agent sent to FDA (courtesy, they ask for it)
REPO_ZIP_PATH = ""                      # optional: path of the repo zip if it is already on the runtime or Drive
REPO_GIT_URL  = ""                      # or a GitHub URL, e.g. https://github.com/YOU/predicate-data-pipeline.git (private repos: use a token URL); skips the upload

# Test 5: K-numbers to download live from FDA (space/newline separated). First 10 have hand-verified predicates; the rest test generalisation.
PROBE_KS = """
K191717 K243175 K121550 K982447 K991039 K991190 K993226 K040237 K942633 K153177
K990380 K984530 K993530 K031564 K033437 K101306 K093244
"""

# Test 6: mini pilot (real openFDA metadata + real PDFs for one review panel)
RUN_PILOT   = True
PILOT_PANEL = "RA"      # RA radiology, CV cardiovascular, OR orthopedic, GU urology, SU general surgery, ...
PILOT_N     = 400       # subject devices (same as run_pilot.sh; the pilot thresholds assume ~400). The predicates they cite are fetched too
                        # (~2.4x more PDFs). Measured on free Colab: 150 subjects = 16 min, so 400 = roughly 40-50 min. Use 150 for a quick look.
RUN_OCR_STRESS = True

## 1. Environment + helpers

In [ ]:
import os, sys, re, json, glob, time, shutil, subprocess, zipfile, pathlib, textwrap, collections
IN_COLAB = "google.colab" in sys.modules
WORK = pathlib.Path("/content" if IN_COLAB else os.environ.get("NB_WORK", "/tmp/nbwork")); WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "predicate-data"
os.environ["CONTACT_EMAIL"] = CONTACT_EMAIL
RESULTS = collections.OrderedDict()          # test name -> (status, detail); printed as a scorecard at the end

def sh(cmd, cwd=None, check=True, quiet=False):
    """Run a shell command, stream its output, return it."""
    p = subprocess.Popen(cmd, shell=True, cwd=str(cwd or (REPO if REPO.exists() else WORK)), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet: print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return "".join(out)

def record(name, status, detail=""):
    RESULTS[name] = (status, detail)
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "SKIP": "⏭️"}.get(status, "•")
    print(f"\n{icon} {name}: {status}  {detail}")

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 200)
print("python", sys.version.split()[0], "| colab:", IN_COLAB, "| cpus:", os.cpu_count())
print(sh("free -g | head -2; df -h " + str(WORK) + " | tail -1", cwd=WORK, quiet=True))

In [ ]:
# system packages: OCR + PDF tools (Colab has root; ~1 min the first time)
missing = [t for t in ("tesseract", "pdftotext", "pdftoppm") if not shutil.which(t)]
if missing:
    sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq tesseract-ocr poppler-utils", cwd=WORK)
print(sh("tesseract --version 2>&1 | head -1; pdftotext -v 2>&1 | head -1", cwd=WORK, quiet=True))

In [ ]:
# get the repo: upload predicate-data.zip (once), unzip, install python deps
if not REPO.exists() and REPO_GIT_URL:
    sh(f"git clone --depth 1 {REPO_GIT_URL} {REPO}", cwd=WORK)
if not REPO.exists():
    zp = REPO_ZIP_PATH if REPO_ZIP_PATH and os.path.exists(REPO_ZIP_PATH) else None
    if not zp:
        found = sorted(glob.glob(str(WORK / "predicate-data*.zip")))
        zp = found[-1] if found else None
    if not zp and IN_COLAB:
        from google.colab import files
        print("Choose predicate-data.zip ...")
        up = files.upload()
        zp = str(WORK / next(iter(up)))
        open(zp, "wb").write(next(iter(up.values())))
    assert zp, "no zip: set REPO_ZIP_PATH or upload predicate-data.zip"
    with zipfile.ZipFile(zp) as z: z.extractall(WORK)
    if not REPO.exists():                      # the zip's top folder may have another name (e.g. predicate-data-pipeline)
        tops = [p for p in WORK.iterdir() if p.is_dir() and (p / "predx").is_dir() and p.name != "predicate-data"]
        if tops: tops[0].rename(REPO)
    assert REPO.exists(), f"expected {REPO} (a folder containing predx/) inside the zip"
os.chdir(REPO); sys.path.insert(0, str(REPO))
sh("pip install -q -r requirements.txt")
sh("ls; echo; cat config/probe_ks.txt | head -5")

## Test 1: unit tests (offline)
Runs the 74 repo tests. Expect `74 passed` (the 10 real-PDF checks run separately in Test 3).

In [ ]:
t0 = time.time()
out = sh("python -m pytest -q --color=no --ignore=tests/test_real_pdfs.py 2>&1 | tail -25", check=False)
m = re.search(r"(\d+) passed", out); bad = re.search(r"(\d+) failed", out)
ok = bool(m) and not bad
record("1 unit tests", "PASS" if ok else "FAIL", f"{m.group(0) if m else 'no result'} in {time.time()-t0:.0f}s")

## Test 2: synthetic end-to-end demo (offline)
Generates fake-but-realistic FDA metadata, text and a few rendered PDFs (a third scanned, so OCR runs), then runs **every** stage
and writes the real output files + audit. The audit will show some WARNs on purpose (synthetic text is long, ~25% of predicates are planted missing).
What matters here: no crash, files exist, the invariants check passes.

In [ ]:
t0 = time.time()
sh(f"{sys.executable} -m predx.cli --config config/demo.yaml --set paths.data_dir=data_demo demo --n 8000 --with-pdfs 8 --no-tokenizers 2>&1 | tail -25", check=False)

def md_table(txt, header_prefix):
    """first markdown table whose header row starts with header_prefix -> DataFrame"""
    lines = txt.splitlines()
    for i, l in enumerate(lines):
        if l.startswith(header_prefix):
            cols = [c.strip() for c in l.strip().strip("|").split("|")]
            rows = []
            for r in lines[i + 2:]:
                if not r.startswith("|"): break
                cells = [c.strip() for c in r.strip().strip("|").split("|")]
                rows.append(cells + [""] * (len(cols) - len(cells)))
            return pd.DataFrame(rows, columns=cols)
    return pd.DataFrame()

def read_audit(path):
    txt = pathlib.Path(path).read_text()
    overall = re.search(r"overall: \*\*(\w+)\*\*", txt).group(1)
    df = md_table(txt, "| status |")
    df["status"] = df["status"].map({"PASS": "✅ PASS", "WARN": "⚠️ WARN", "FAIL": "❌ FAIL"}).fillna(df["status"])
    return txt, overall, df

In [ ]:
os.chdir(REPO)
atxt, overall, chk = read_audit(REPO / "data_demo/reports/audit.md")
print("audit overall:", overall); display(chk)
files = sorted(p.name for p in (REPO / "data_demo/dataset").glob("*"))
print("dataset files:", files)
inv = chk[chk["check"].str.contains("well-formed")]
inv_ok = (not inv.empty) and inv.iloc[0]["status"].endswith("PASS")
need = {"corpus.jsonl.gz", "queries_train.jsonl.gz", "queries_dev.jsonl.gz", "queries_test.jsonl.gz", "train_pairs.jsonl.gz", "dataset_card.json"}
record("2 synthetic end-to-end", "PASS" if inv_ok and need <= set(files) else "FAIL",
       f"audit={overall}, invariants={'ok' if inv_ok else 'BROKEN'}, files={'all' if need <= set(files) else 'MISSING ' + str(need - set(files))}, {time.time()-t0:.0f}s")

## Test 3: the 10 real FDA PDFs in the repo (offline)
`tests/real_pdfs/` has 10 real 510(k) documents (1994-2025, digital and scanned) and `expected.json` with the predicates **I read by hand** for 7 of them.
This is the check that parsing real documents still gives the right answers after any rule change.

In [ ]:
out = sh("python -m pytest tests/test_real_pdfs.py -q --color=no 2>&1 | tail -6", check=False)
m = re.search(r"(\d+) passed", out); bad = re.search(r"(\d+) failed", out)
sh("python -m predx.cli probe --pdf-dir tests/real_pdfs --out data_realpdfs > /dev/null 2>&1", check=False)
real = pd.read_csv("data_realpdfs/probe_summary.csv")
display(real[["k", "pages", "ocr_pages", "ifu_words", "desc_words", "primary", "predicates", "warnings"]])
record("3 real-PDF regression", "PASS" if m and not bad else "FAIL", m.group(0) if m else out[-200:])

## Test 4: OCR stress curve (offline, optional)
Degrades synthetic scans at 4 levels and measures how much of the text, K-numbers and predicates survive. L0 = clean; L1 = typical fax/scan; L2 = bad; L3 = very bad (expected to fail).

In [ ]:
if RUN_OCR_STRESS:
    sh("python -m predx.cli ocr-stress --n 4 --levels 0,1,2 2>&1 | grep -v INFO | tail -12", check=False)
    st = pd.read_csv("data_stress/ocr_stress.csv")
    display(st[["level", "docs", "errors", "ifu_ok", "desc_ok", "k_token_accuracy", "primary_ok", "gold_set_exact", "sec_per_doc"]])
    l01 = st[st["level"].isin([0, 1])]
    good = (l01["errors"].sum() == 0) and (l01["gold_set_exact"] >= 0.9).all()
    record("4 OCR stress", "PASS" if good else "FAIL", "clean + typical scans: gold set exact >= 90%, no errors" if good else "levels 0/1 below 90% or errors: see table")
else:
    record("4 OCR stress", "SKIP", "RUN_OCR_STRESS=False")

## Network check: can this runtime reach FDA?
Tests 5-6 need `api.fda.gov`, `download.open.fda.gov` and `www.accessdata.fda.gov`. Colab normally allows these.

In [ ]:
import requests
UA = {"User-Agent": f"predx-course-project/0.1 (academic research; contact: {CONTACT_EMAIL})"}
checks = {
  "openFDA API": "https://api.fda.gov/device/510k.json?limit=1",
  "openFDA bulk index": "https://api.fda.gov/download.json",
  "FDA PDF (K243175)": "https://www.accessdata.fda.gov/cdrh_docs/pdf24/K243175.pdf",
}
NETWORK_OK = True
for name, url in checks.items():
    try:
        r = requests.get(url, headers=UA, timeout=40, stream=True)
        head = next(r.iter_content(8), b"")
        ok = r.status_code == 200 and (head.startswith(b"%PDF") if url.endswith(".pdf") else True)
        print(("✅" if ok else "❌"), name, r.status_code, r.headers.get("content-type", ""))
        NETWORK_OK &= ok
    except Exception as e:
        print("❌", name, type(e).__name__, str(e)[:120]); NETWORK_OK = False
print("\nNETWORK_OK =", NETWORK_OK, "" if NETWORK_OK else " -> tests 5-6 will be skipped (try again later; FDA's server is sometimes slow)")

## Test 5: live FDA probe
Downloads the PDFs in `PROBE_KS` straight from FDA with the real fetcher (this tests URL folder guessing for every era), runs the real extraction + parsing,
and **scores itself** against the hand-verified predicates in `tests/real_pdfs/expected.json` for the K-numbers that have them.
For the rest, use the viewer cell below to check by eye.

In [ ]:
PROBE = None
if NETWORK_OK:
    DEFAULT_KS = "K191717 K243175 K121550 K982447 K991039 K991190 K993226 K040237 K942633 K153177 K990380 K984530 K993530 K031564 K033437 K101306 K093244".split()
    ks = PROBE_KS.split() or DEFAULT_KS
    if not PROBE_KS.split(): print("PROBE_KS is empty -> using the default list of", len(ks), "K-numbers")
    pathlib.Path("/tmp/probe_ks.txt").write_text("\n".join(ks) + "\n")
    t0 = time.time()
    sh("python -m predx.cli --set paths.data_dir=data_probe_run probe --ks-file /tmp/probe_ks.txt --out data_probe_live 2>&1 | tail -40", check=False)
    PROBE = pd.read_csv("data_probe_live/probe_summary.csv")
    fetched = PROBE[PROBE["chars"] > 0]
    exp = json.loads(pathlib.Path("tests/real_pdfs/expected.json").read_text())
    rows = []
    for k, e in exp.items():
        r = PROBE[PROBE["k"] == k]
        got = (r.iloc[0]["primary"] if len(r) and isinstance(r.iloc[0]["primary"], str) else "")
        rows.append({"k": k, "expected_primary": e["primary"], "got_primary": got or "(none)", "match": "✅" if got == e["primary"] else "❌"})
    score = pd.DataFrame(rows)
    print(f"\nfetched+parsed: {len(fetched)}/{len(ks)}   time {time.time()-t0:.0f}s")
    display(PROBE[["k", "pages", "ocr_pages", "ifu_words", "desc_words", "primary", "n_warnings", "warnings"]])
    display(score)
    fetch_rate = len(fetched) / len(ks)
    acc = (score["match"] == "✅").mean() if len(score) else float("nan")
    status = "PASS" if fetch_rate >= 0.9 and acc >= 0.85 else ("WARN" if fetch_rate >= 0.7 else "FAIL")
    record("5 live FDA probe", status, f"fetched {fetch_rate:.0%}, primary predicate matches hand labels on {acc:.0%} of {len(score)}")
else:
    record("5 live FDA probe", "SKIP", "FDA not reachable from this runtime")

In [ ]:
# Viewer: compare what the pipeline extracted with the actual PDF. Change K and re-run as often as you like.
from IPython.display import display, Image
from predx.probe import probe_pdf
from predx.config import load_config

def find_pdf(k):
    hits = list(REPO.rglob(f"{k}.pdf"))
    return hits[0] if hits else None

def render(pdf, pages, dpi=70):
    for p in pages:
        out = f"/tmp/pg_{pdf.stem}_{p}"
        subprocess.run(["pdftoppm", "-f", str(p), "-l", str(p), "-r", str(dpi), "-png", "-singlefile", str(pdf), out], check=False)
        if os.path.exists(out + ".png"): display(Image(filename=out + ".png", width=520))

def view(k, pages=(1,), show_text=True):
    pdf = find_pdf(k)
    if not pdf: print("PDF not found for", k); return
    r = probe_pdf(pdf, k, load_config())
    p = r["parsed"]
    print(f"{k}  pages={r.get('n_pages')}  ocr_pages={len(r.get('ocr_pages', []))}  warnings={[w.split(':')[0] for w in r['warnings']]}")
    print("IFU        :", (p.get("ifu") or "(none)")[:300])
    print("DESCRIPTION:", (p.get("description") or "(none)")[:300].replace("\n", " "), "...", f"[{p.get('desc_source')}, {p.get('desc_words')} words]")
    print("PREDICATES :", [(x["k"], x["role"], f"p.{x['page']+1}") for x in p["predicates"]] or "none")
    print("FDA page   : https://www.accessdata.fda.gov/scripts/cdrh/cfdocs/cfpmn/pmn.cfm?ID=" + k)
    render(pdf, pages)

view("K993226", pages=(1,))     # <- change this K-number / pages

## Test 6: mini pilot on real data
`pilot` = download the openFDA metadata (all 176k records, one bulk file) → sample `PILOT_N` subjects of one review panel, stratified by year → fetch + extract + parse them **and the predicates they cite** →
build the dataset (corpus, queries, splits, pairs, leakage masking) → validate → audit.
The audit needs a few hundred subjects to pass its *size* checks, so with a small pilot the size checks FAIL/WARN by design; the **rate** checks (download, text, indications, predicates found, no leakage) are what to judge. The verdict cell below only uses those.

In [ ]:
PILOT_DIR = REPO / "data_pilot_run"
if RUN_PILOT and NETWORK_OK:
    t0 = time.time()
    sh(f"python -m predx.cli --config config/pilot.yaml --set paths.data_dir=data_pilot_run pilot --panel {PILOT_PANEL} --n {PILOT_N} --no-baselines --no-tokenizers 2>&1 | tail -45", check=False)
    print(f"\npilot wall time: {(time.time()-t0)/60:.1f} min")
else:
    print("pilot skipped (RUN_PILOT=False or no network)")

### Evaluate the pilot (automatic)
The verdict looks only at the *rate* checks that do not depend on how many subjects you sampled.

In [ ]:
CORE = ["well-formed", "PDF download success", "text extraction success", "indications", "extracted predicate",
        "no K-numbers left", "naming a cited predicate"]
ap = PILOT_DIR / "reports/audit.md"
if ap.exists():
    atxt, overall, chk = read_audit(ap)
    print("audit overall:", overall, "(size checks FAIL/WARN is normal for a small pilot)\n")
    display(chk)
    core = chk[chk["check"].apply(lambda c: any(w in c for w in CORE))]
    bad = core[~core["status"].str.endswith("PASS")]
    print("\n--- core rate checks ---"); display(core)
    print("--- funnel ---"); display(md_table(atxt, "| step |").head(12))
    yr = md_table(atxt, "| year | n | pdf_rate")
    if len(yr): print("--- by year ---"); display(yr)
    ocr = [l for l in atxt.splitlines() if "seconds/doc" in l or "mean seconds" in l]
    print("\n".join(ocr[:3]))
    status = "PASS" if bad.empty else ("FAIL" if bad["status"].str.contains("FAIL").any() else "WARN")
    record("6 mini pilot (core rates)", status, "all core checks pass" if bad.empty else "check: " + "; ".join(bad["check"].str[:50]))
else:
    record("6 mini pilot (core rates)", "SKIP", "no pilot audit found")

In [ ]:
# peek at the actual dataset the pilot produced (this is the format training will read)
import gzip
D = PILOT_DIR / "dataset"
if D.exists():
    def head(f, n=1):
        with gzip.open(D / f, "rt") as fh: return [json.loads(next(fh)) for _ in range(n)]
    c = head("corpus.jsonl.gz")[0]; c["ifu"] = c["ifu"][:80] + "..."; c["description"] = c["description"][:80] + "..."
    print("CORPUS record  :", json.dumps(c)[:600])
    for split in ("train", "dev", "test"):
        f = D / f"queries_{split}.jsonl.gz"
        if f.exists():
            n = sum(1 for _ in gzip.open(f, "rt"))
            print(f"queries_{split}: {n} records")
    q = head("queries_train.jsonl.gz")[0] if (D / "queries_train.jsonl.gz").exists() else None
    if q: q["ifu"] = q["ifu"][:60] + "..."; q["description"] = q["description"][:60] + "..."; print("QUERY record   :", json.dumps(q)[:900])
    print("dataset_card   :", json.dumps({k: v for k, v in json.load(open(D / "dataset_card.json")).items() if k in ("n_meta", "n_corpus", "counts", "mask_level")}))

## Hand check (the number that matters most)
The audit can only tell you that extraction *ran*. This step tells you whether the predicates are *right*:
1. `label-sample` picks documents (30% from risky ones: ambiguous, no predicate found, or OCR'd).
2. For each, `review(k)` shows what the pipeline extracted next to the PDF page where it found it. Open the FDA page link if you need the full document.
3. Mark the right answer: `confirm([...])` if the extraction is correct, `correct(k, primary=..., additional=..., reference=...)` if not.
4. `label-eval` scores precision, recall and primary-predicate accuracy. **Bar I would use: both ≥ ~0.90** (my judgement, not a number from the proposal).
Aim for 20-30 documents here (the full 100 comes on the real run).

In [ ]:
import csv
LAB = PILOT_DIR / "labeling"
sample = []
if (PILOT_DIR / "parsed.jsonl.gz").exists():
    sh(f"python -m predx.cli --config config/pilot.yaml --set paths.data_dir=data_pilot_run label-sample --n 25 2>&1 | tail -3", check=False)
    sample = [r["k"] for r in csv.DictReader(open(LAB / "label_sample.csv"))]
print(len(sample), "documents:", sample)

def _load(): return pd.read_csv(LAB / "label_sample.csv", dtype=str).fillna("")
def _save(df): df.to_csv(LAB / "label_sample.csv", index=False)

def review(k, dpi=70):
    df = _load(); r = df[df.k == k].iloc[0]
    print(f"=== {k}  year={r.year}  code={r.product_code}  ocr_pages={r.ocr_pages}  flagged={r.flagged}")
    print("extracted  primary:", r.extracted_primary or "-", "| additional:", r.extracted_additional or "-", "| reference:", r.extracted_reference or "-", "| other:", r.extracted_other or "-")
    snip = LAB / "snippets" / f"{k}.txt"
    if snip.exists(): print(snip.read_text()[:1200])
    pages = sorted({int(p) for p in re.findall(r"p\.(\d+)", snip.read_text())})[:2] if snip.exists() else []
    pdf = find_pdf(k)
    print("FDA page: https://www.accessdata.fda.gov/scripts/cdrh/cfdocs/cfpmn/pmn.cfm?ID=" + k)
    if pdf: render(pdf, pages or [1], dpi)

def confirm(ks):
    """the extraction is right for these documents"""
    df = _load()
    for k in ks:
        i = df.index[df.k == k]
        df.loc[i, "hand_primary"] = df.loc[i, "extracted_primary"]; df.loc[i, "hand_additional"] = df.loc[i, "extracted_additional"]
        df.loc[i, "hand_reference"] = df.loc[i, "extracted_reference"]; df.loc[i, "done"] = "y"
    _save(df); print("confirmed", len(ks))

def correct(k, primary="", additional="", reference="", note=""):
    """the true answer (K-numbers separated by ;). Use '' for none."""
    df = _load(); i = df.index[df.k == k]
    df.loc[i, ["hand_primary", "hand_additional", "hand_reference", "notes", "done"]] = [primary, additional, reference, note, "y"]
    _save(df); print("saved", k)

review(sample[0]) if sample else None

In [ ]:
# work through the sample: look, then confirm / correct. Example (edit!):
# review(sample[1]); review(sample[2])
# confirm([sample[0], sample[1], sample[2]])
# correct(sample[3], primary="K123456", additional="K234567;K345678", note="pipeline picked the wrong primary")
for k in sample[1:4]:
    review(k)

In [ ]:
# score your labels (needs at least a few documents marked done)
if sample:
    out = sh("python -m predx.cli --config config/pilot.yaml --set paths.data_dir=data_pilot_run label-eval 2>&1 | tail -30", check=False)
    done = int((_load()["done"].str.lower().isin(["y", "yes", "1", "x"])).sum())
    rec = re.search(r"gold set[^\n]*recall (\d\.\d+)", out); prim = re.search(r"primary accuracy: (\d\.\d+)", out)
    if done < 15 or not (rec and prim):
        record("hand check", "SKIP", f"only {done} documents labelled; label at least 15 (confirm / correct above), then re-run this cell")
    else:
        r_, p_ = float(rec.group(1)), float(prim.group(1))
        record("hand check", "PASS" if min(r_, p_) >= 0.90 else "FAIL", f"{done} labelled: gold-set recall {r_:.2f}, primary accuracy {p_:.2f} (bar ~0.90)")

## Scorecard + keep your results

In [ ]:
rows = [{"test": k, "status": {"PASS": "✅ PASS", "WARN": "⚠️ WARN", "FAIL": "❌ FAIL", "SKIP": "⏭️ SKIP"}[v[0]], "detail": v[1]} for k, v in RESULTS.items()]
display(pd.DataFrame(rows))
n_fail = sum(v[0] == "FAIL" for v in RESULTS.values())
print("\nOVERALL:", "❌ fix the FAILs before running at scale" if n_fail else "✅ nothing failed; read the WARNs and finish the hand check before the full run")

In [ ]:
# zip the reports + a small dataset sample to download (the temporary disk disappears when the runtime ends)
bundle = WORK / "predx_colab_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ["data_demo/reports", "data_probe_live", "data_realpdfs/probe_report.md", "data_pilot_run/reports", "data_pilot_run/labeling",
              "data_pilot_run/dataset/dataset_card.json", "data_pilot_run/device_status.jsonl.gz" ]:
        p = REPO / d
        if p.is_file(): z.write(p, d)
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file() and f.suffix != ".pdf" and f.stat().st_size < 30e6: z.write(f, str(f.relative_to(REPO)))
print(bundle, f"{bundle.stat().st_size/1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))
    # optional: keep everything incl. pilot data on Drive
    # from google.colab import drive; drive.mount('/content/drive'); !cp -r /content/predicate-data/data_pilot_run /content/drive/MyDrive/predx_pilot

## If something fails
| symptom | likely cause | what to do |
|---|---|---|
| `NETWORK_OK = False` | FDA server slow / throttling | wait a minute and re-run the network cell; free Colab sometimes blocks nothing, FDA sometimes times out |
| many `FETCH_FAILED` in Test 5 | wrong PDF folder for an era, or rate limit | send me the probe table; the fetcher tries `pdf`, `pdfYY`, `pdfY` in order |
| Test 6 stops with a memory error | metadata bulk file + parse in 12 GB | lower `PILOT_N`; runtime → restart and run from the top (cache is kept if the disk survived) |
| runtime disconnected | idle > ~90 min | reconnect and re-run all cells; stages skip finished work if files survived, otherwise it redoes them |
| audit shows FAIL on `min_subjects` etc. | pilot too small | expected; judge only the core rate checks |
| low primary-predicate accuracy in the hand check | a parsing rule is wrong | send me 3-5 of the wrong K-numbers; fix is usually one regex, and the real PDFs become regression tests |